# Subword Tokenization — BPE, WordPiece, Unigram, SentencePiece Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: BPE from scratch

See `code/main.py`. The loop:

In [ ]:
```python

def train_bpe(corpus, num_merges):

    vocab = {tuple(word) + ("</w>",): count for word, count in corpus.items()}

    merges = []

    for _ in range(num_merges):

        pairs = Counter()

        for symbols, freq in vocab.items():

            for a, b in zip(symbols, symbols[1:]):

                pairs[(a, b)] += freq

        if not pairs:

            break

        best = pairs.most_common(1)[0][0]

        merges.append(best)

        vocab = apply_merge(vocab, best)

    return merges

In [ ]:
```

Three facts the algorithm encodes. `</w>` marks word end so "low" (suffix) and "lower" (prefix) stay distinct. Frequency weighting makes high-frequency pairs win early. The merge list is ordered — inference applies merges in training order.

### Step 2: encode with the learned merges

In [ ]:
```python

def encode_bpe(word, merges):

    symbols = list(word) + ["</w>"]

    for a, b in merges:

        i = 0

        while i < len(symbols) - 1:

            if symbols[i] == a and symbols[i + 1] == b:

                symbols = symbols[:i] + [a + b] + symbols[i + 2:]

            else:

                i += 1

    return symbols

In [ ]:
```

Naive O(n·|merges|). Production implementations (tiktoken, HF Tokenizers) use merge-rank lookup with priority queues and run in near-linear time.

### Step 3: SentencePiece in practice

In [ ]:
```python

import sentencepiece as spm

spm.SentencePieceTrainer.train(

    input="corpus.txt",

    model_prefix="my_tokenizer",

    vocab_size=8000,

    model_type="bpe",          # or "unigram"

    character_coverage=0.9995, # lower for CJK (e.g. 0.9995 for English, 0.995 for Japanese)

    normalization_rule_name="nmt_nfkc",

)

sp = spm.SentencePieceProcessor(model_file="my_tokenizer.model")

print(sp.encode("untokenizable", out_type=str))

# ['▁un', 'token', 'izable']

In [ ]:
```

Notice: no pre-tokenization required, space encoded as `▁`, `character_coverage` controls how aggressively rare characters are preserved vs mapped to `<unk>`.

### Step 4: tiktoken for OpenAI-compatible vocabs

In [ ]:
```python

import tiktoken

enc = tiktoken.get_encoding("o200k_base")

print(enc.encode("untokenizable"))        # [127340, 101028]

print(len(enc.encode("Hello, world!")))   # 4

In [ ]:
```

Encoding-only. Fast (Rust backend). Exact match with GPT-4/5 tokenization for byte-counting, cost estimation, context-window budgeting.

## Exercises

In [ ]:
1. **Easy.** Train a 500-merge BPE on `code/main.py`'s tiny corpus. Encode three held-out words. How many produced exactly 1 token vs >1 token?
2. **Medium.** Compare token counts on 100 English Wikipedia sentences between `cl100k_base`, `o200k_base`, and a SentencePiece BPE you train with vocab=32k. Report the compression ratio of each.
3. **Hard.** Train the same corpus with BPE, Unigram, and WordPiece. Measure downstream accuracy when using each on a small sentiment classifier. Does the choice move the needle by more than 1 point F1?